# Практика · GAN

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ Зошит навчає одного суддю, **девʼять** GAN (три конфігурації на трьох зернах) і
> **три** VAE. Заміряно на чотирьох ядрах без відеокарти, серійним прогоном:
> **близько чотирьох хвилин** — у шести прогонах вийшло від 208 до 376 секунд, і
> розкид дає завантаження машини, а не сам зошит. Чотири пʼятих цього часу займає
> навчання девʼяти GAN; воно й друкує рядок «девʼять GAN навчено за … с», тож
> звіряйся з ним. Рахунок іде в один потік — на таких дрібних мережах так і швидше,
> і відтворюваніше.

Тут кожне твердження лекції перетворюється на число.

1. **Насичена й ненасичена втрата генератора** — рахуємо градієнт руками й
   звіряємо з `torch.autograd`.
2. **Суддя** — класифікатор шести фігур, і перевірка, що на справжніх даних йому легко.
3. **Калібрування метрики** — вісім завідомо поганих наборів. Метрика, яка не
   поставить «лише кола» гірше за робочу модель, до роботи не допускається.
4. **Колапс мод** — скільки з шести фігур доживає до кінця навчання.
5. **Три конфігурації на трьох зернах** — і перевірка найпоширенішої поради
   «послаб дискримінатора».
6. **Криві втрат** — і доказ, що за ними не видно нічого.
7. **Чому першим гине кільце** — замір крихкості фігур.
8. **GAN проти VAE** — однією метрикою: різкість проти покриття.

**Мережа не потрібна:** усі зображення ми малюємо формулами.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy
from scipy import linalg

# зерна фіксуємо на самому початку: без них числа нижче не збіжаться з лекцією
torch.manual_seed(0)

# один потік, а не чотири. Мережі тут крихітні, і чотири потоки більше часу
# домовляються між собою, ніж рахують. Друга причина важливіша за швидкість:
# під кількома потоками float-суми йдуть в іншому порядку, і числа пливуть.
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("scipy      :", scipy.__version__)
print("потоків CPU:", torch.get_num_threads())

## 1 · Дві втрати генератора: рахуємо градієнт руками

Найдешевший замір теми — він не потребує ані даних, ані навчання, і при цьому
пояснює рядок коду, який стоїть у кожній реалізації GAN.

Дискримінатор видає одне число `s` (його звуть **логіт**), а ймовірність «це
справжнє» дістають із нього сигмоїдою: `D = 1 / (1 + exp(-s))`.

Оригінальна стаття пропонує генераторові **мінімізувати** `log(1 - D)` — тобто
знижувати впевненість дискримінатора в тому, що підробка фальшива. На практиці
так ніхто не робить. На початку навчання дискримінатор легко розрізняє, `D`
близьке до нуля — і градієнт цієї втрати теж близький до нуля. Генератор стоїть
саме тоді, коли йому найбільше треба вчитись.

Замість цього генератор **мінімізує** `-log D`. Точка оптимуму та сама, а
поведінка градієнта протилежна. Похідні по логіту виводяться в один рядок:

* насичена: похідна `log(1 - sigmoid(s))` по `s` дорівнює `-sigmoid(s)`, тобто `-D`;
* ненасичена: похідна `-log sigmoid(s)` по `s` дорівнює `sigmoid(s) - 1`, тобто `D - 1`.

Отже модуль градієнта — це `D` у першому випадку й `1 - D` у другому. Порахуємо
обидва числа й подивимось, у скільки разів вони різняться.

In [ ]:
def sigmoid(s):
    return 1.0 / (1.0 + np.exp(-s))


print("%-9s %-10s %-15s %-16s %s" %
      ("логіт s", "D = σ(s)", "|град| насич.", "|град| ненасич.", "у скільки разів"))
print("-" * 68)

saturation_table = []
for logit in [-4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0]:
    d_value = sigmoid(logit)
    grad_saturating = d_value              # модуль від -D
    grad_nonsaturating = 1.0 - d_value     # модуль від D - 1
    saturation_table.append((logit, d_value, grad_saturating, grad_nonsaturating))
    print("%-9.1f %-10.4f %-15.4f %-16.4f %.1f" %
          (logit, d_value, grad_saturating, grad_nonsaturating,
           grad_nonsaturating / grad_saturating))

Тепер обовʼязкова звірка «наша реалізація проти бібліотечної»: попросимо
`torch.autograd` порахувати ті самі похідні автоматично.

In [ ]:
logits = torch.tensor([-4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
loss_saturating = torch.log(1 - torch.sigmoid(logits)).sum()
grad_saturating_torch = torch.autograd.grad(loss_saturating, logits)[0].abs().numpy()

logits_again = torch.tensor([-4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
loss_nonsaturating = (-torch.log(torch.sigmoid(logits_again))).sum()
grad_nonsaturating_torch = torch.autograd.grad(loss_nonsaturating, logits_again)[0].abs().numpy()

by_hand_saturating = np.array([row[2] for row in saturation_table])
by_hand_nonsaturating = np.array([row[3] for row in saturation_table])

assert np.allclose(by_hand_saturating, grad_saturating_torch, atol=1e-6), "насичена розійшлася!"
assert np.allclose(by_hand_nonsaturating, grad_nonsaturating_torch, atol=1e-6), "ненасичена розійшлася!"

print("руками  :", np.round(by_hand_saturating, 4))
print("autograd:", np.round(grad_saturating_torch, 4))
print("✅ збігається — і для насиченої, і для ненасиченої")

Головний рядок таблиці — перший. При `s = -4` дискримінатор дає `D = 0.0180`: він
майже впевнений, що бачить підробку. Насичена втрата має тут градієнт **0.0180**,
ненасичена — **0.9820**, тобто у **54.6 раза** більший. Через це в кожній реальній
реалізації GAN стоїть ненасичений варіант, і ми теж беремо його.

## 2 · Датасет: ті самі шість фігур

Наскрізний приклад усього блоку — шість фігур 28×28 із випадковим центром,
випадковим радіусом і шумом σ = 0.06. Генератор фігур дослівно той самий, що в
темі [16 «Самонаглядове навчання»](../16-self-supervised/lecture.html), тільки шум
менший: намалювати фігуру складніше, ніж упізнати, і зайвий шум заважає читати
результат.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=5, noise=0.06, center=None, radius=None):
    # Малює одну фігуру як масив 28 на 28 зі значеннями 0..1.
    image = np.zeros((size, size), dtype=np.float32)
    if center is None:
        center_y = size / 2 + rng.integers(-jitter, jitter + 1)
        center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    else:
        center_y, center_x = center
    if radius is None:
        radius = rng.integers(5, 9)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    if noise > 0:
        image = image + rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_dataset(count, rng):
    # Повертає (count, 1, 28, 28) і (count,). Класів шість, порівну.
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % 6                       # рівно по шостій частині кожного класу
        images[i, 0] = draw_shape(kind, rng)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


rng = np.random.default_rng(42)
x_train, y_train = make_dataset(1800, rng)          # на цьому вчаться і суддя, і GAN
x_test, y_test = make_dataset(600, rng)             # перевірка судді
x_reference, y_reference = make_dataset(600, rng)   # еталонна купа для FID

print("навчальні :", tuple(x_train.shape))
print("перевірні :", tuple(x_test.shape))
print("еталон FID:", tuple(x_reference.shape))
print("частка кожного класу: %.4f" % (1 / 6))

Подивимось на всі шість фігур очима й порахуємо, скільки в кожній «намальованих»
пікселів. Це число знадобиться пізніше, коли ми питатимемо, чому одні класи гинуть
раніше за інших.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 6, figsize=(10, 2.0))
for kind in range(6):
    axes[kind].imshow(x_train[kind, 0], cmap="gray", vmin=0, vmax=1)
    axes[kind].set_title(SHAPE_NAMES[kind], fontsize=9)
    axes[kind].axis("off")
plt.tight_layout()
plt.show()

print("%-11s %s" % ("фігура", "частка яскравих пікселів"))
bright_share = []
for kind in range(6):
    same_class = x_train[kind::6, 0].numpy()
    share = float((same_class > 0.5).mean())
    bright_share.append(share)
    print("%-11s %.4f" % (SHAPE_NAMES[kind], share))

## 3 · Суддя

Згенеровану картинку не можна оцінити «на око» в тексті, де числа мусить друкувати
зошит. Тому нам потрібен **суддя** — окремий класифікатор шести фігур, навчений на
справжніх даних.

У нього три ролі:
* сказати, **яка це фігура** — з цього ми порахуємо покриття класів;
* дати **впевненість** — і показати, чому їй не можна вірити;
* дати **64 числа ознак** із передостаннього шару — на них будуватимемо FID.

In [ ]:
def conv_block(in_channels, out_channels):
    # Один типовий блок: Conv → ReLU → Pool. Той самий, що в темах 12 і 16.
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class Judge(nn.Module):
    # Класифікатор фігур. Шар squeeze дає 64 числа — саме їх бере FID.
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(conv_block(1, 8), conv_block(8, 16),
                                  conv_block(16, 32), nn.Flatten())
        self.squeeze = nn.Linear(288, 64)
        self.head = nn.Linear(64, 6)

    def features(self, x):
        return F.relu(self.squeeze(self.body(x)))

    def forward(self, x):
        return self.head(self.features(x))


def train_judge(images, labels, seed=0, epochs=8, batch_size=64, lr=3e-3):
    torch.manual_seed(seed)
    model = Judge()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    count = len(images)
    for epoch in range(epochs):
        order = torch.randperm(count)
        for start in range(0, count, batch_size):
            batch = order[start:start + batch_size]
            optimizer.zero_grad()
            F.cross_entropy(model(images[batch]), labels[batch]).backward()
            optimizer.step()
    model.eval()
    return model


judge_started = time.perf_counter()
judge = train_judge(x_train, y_train)
print("суддя навчений за %.1f с" % (time.perf_counter() - judge_started))

with torch.no_grad():
    judge_accuracy = float((judge(x_test).argmax(1) == y_test).float().mean())
print("точність судді на справжніх фігурах: %.4f" % judge_accuracy)

Задача для судді тривіальна — і це саме те, що нам треба. Якщо на справжніх фігурах
він майже не помиляється, то будь-яка його невпевненість на згенерованому щось
означає.

Тепер два інструменти, якими ми міритимемо генерацію.

**Покриття класів.** Проганяємо 600 згенерованих картинок крізь суддю й дивимось,
скільки з шести класів отримали хоч 5 % вибірки. Максимум — 6.

**FID.** Відстань Фреше між двома купами ознак: рахуємо середнє й коваріаційну
матрицю для справжніх і для згенерованих, і міряємо, наскільки різні ці два хмарні
описи. Формула нижче — точна відстань між двома гаусіанами.

> ⚠️ Наш FID **не порівнюваний з опублікованими числами**. Справжній FID рахують на
> ознаках Inception, навченого на ImageNet; ми беремо ознаки власного судді, який
> знає рівно шість фігур. Наше число годиться, щоб порівнювати наші моделі між
> собою, і не годиться, щоб цитувати поруч зі статтями.

In [ ]:
@torch.no_grad()
def judge_report(model, images):
    # Повертає частки шести класів і середню впевненість.
    probabilities = F.softmax(model(images), dim=1)
    confidence, predicted = probabilities.max(1)
    shares = np.bincount(predicted.numpy(), minlength=6) / len(images)
    return shares, float(confidence.mean())


@torch.no_grad()
def feature_cloud(model, images, batch_size=256):
    # 64 числа на картинку — та купа, між якими рахується відстань Фреше.
    parts = []
    for start in range(0, len(images), batch_size):
        parts.append(model.features(images[start:start + batch_size]).numpy())
    return np.concatenate(parts)


def frechet_distance(cloud_a, cloud_b, eps=1e-6):
    # |μa - μb|² + tr(Σa + Σb - 2·(Σa·Σb)^(1/2))
    mean_a, mean_b = cloud_a.mean(0), cloud_b.mean(0)
    # частина з 64 ознак завжди нульова (ReLU їх погасив), і коваріаційна
    # матриця через це вироджена. Крихітна добавка на діагональ робить корінь
    # обчислюваним і майже не змінює число — так роблять і в справжніх FID
    regulariser = eps * np.eye(cloud_a.shape[1])
    cov_a = np.cov(cloud_a, rowvar=False) + regulariser
    cov_b = np.cov(cloud_b, rowvar=False) + regulariser
    difference = mean_a - mean_b
    root = linalg.sqrtm(cov_a.dot(cov_b))
    if np.iscomplexobj(root):
        root = root.real
    return float(difference.dot(difference) + np.trace(cov_a) + np.trace(cov_b)
                 - 2 * np.trace(root))


def midtone_share(images):
    # частка пікселів, які ані чорні, ані білі: найпростіша міра розмитості
    values = images.numpy()
    return float(((values > 0.25) & (values < 0.75)).mean())


reference_cloud = feature_cloud(judge, x_reference)


def score(images):
    # Три числа на кожен набір: покриття, FID і впевненість судді.
    shares, confidence = judge_report(judge, images)
    covered = int((shares >= 0.05).sum())
    return covered, frechet_distance(feature_cloud(judge, images), reference_cloud), confidence, shares


# перевірка формули: відстань купи до себе самої мусить бути нулем
self_distance = abs(frechet_distance(reference_cloud, reference_cloud))
assert self_distance < 1e-4, "відстань до себе не нуль — формула зламана"
print("FID купи до себе самої: %.6f  ✅ нуль, як і має бути" % self_distance)

## 4 · Калібрування: спершу перевіряємо метрику, потім модель

Це найважливіша клітинка зошита, і пропустити її не можна.

Метрика генерації — не термометр, який просто працює. Вона може мовчки міряти не
те, і тоді всі числа, поміряні нею, доведеться викинути. Тому перед першим
використанням її проганяють через набір **завідомо поганих** зразків.

Головний рядок майбутньої таблиці — «лише кола». Це повний колапс мод: один клас із
шести. Метрика, яка ставить його **краще** за робочу модель, до роботи не
допускається.

In [ ]:
calibration_rng = np.random.default_rng(7)


def only_kinds(kinds, count=600):
    # Набір, у якому є лише перелічені класи, — модель колапсу мод.
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    for i in range(count):
        images[i, 0] = draw_shape(kinds[i % len(kinds)], calibration_rng)
    return torch.from_numpy(images)


# «розмита каша»: справжні фігури, двічі згладжені вікном 5 на 5
blurred = F.avg_pool2d(x_test, 5, stride=1, padding=2)
blurred = F.avg_pool2d(blurred, 5, stride=1, padding=2)

calibration_sets = [
    ("справжні, інша вибірка", x_test),
    ("лише кола", only_kinds([0])),
    ("кола + квадрати", only_kinds([0, 1])),
    ("розмита каша", blurred),
    ("середнє всіх фігур", x_train.mean(0, keepdim=True).repeat(600, 1, 1, 1)),
    ("усе чорне", torch.zeros(600, 1, 28, 28)),
    ("чистий шум", torch.rand(600, 1, 28, 28)),
    ("усе біле", torch.ones(600, 1, 28, 28)),
]

print("%-24s %8s %12s %10s" % ("що подаємо", "класів", "FID", "впевненість"))
print("-" * 58)
calibration_rows = []
for name, images in calibration_sets:
    covered, fid, confidence, _ = score(images)
    calibration_rows.append((name, covered, fid, confidence))
    print("%-24s %8d %12.4f %10.4f" % (name, covered, fid, confidence))

Читай перший рядок і другий разом.

**Справжні фігури з іншої вибірки** дають FID близько одиниці — це базова лінія,
шум оцінки, а не відстань. Друкувати її поруч обовʼязково, інакше читач сприйме за
нуль будь-яке мале число.

**«Лише кола»** дають FID у сотні разів більший і покриття 1. Метрика впоралася:
повний колапс мод вона ставить дуже погано. Саме цієї перевірки не витримала
слабша версія метрики, і в блоці вже був випадок, коли «лише кола» опинилися
**краще** за робочу модель.

І звернімо увагу на останній стовпчик уже тут: на «всьому чорному» суддя впевнений
сильніше, ніж на справжніх фігурах. Впевненість — не міра якості, і далі ми
побачимо це ще двічі.

In [ ]:
real_baseline_fid = calibration_rows[0][2]
circles_only_fid = calibration_rows[1][2]
print("базова лінія (справжні проти справжніх): %.4f" % real_baseline_fid)
print("повний колапс мод (лише кола)          : %.4f" % circles_only_fid)
print("у скільки разів гірше                  : %.1f" % (circles_only_fid / real_baseline_fid))
assert circles_only_fid > 50 * real_baseline_fid, "метрика сліпа до колапсу — далі йти не можна"
print("✅ метрика бачить колапс мод — можна міряти")

## 5 · Крихітний DCGAN

Тепер самі мережі. Генератор бере 16 випадкових чисел і робить із них картинку
28×28; дискримінатор бере картинку й видає одне число.

Архітектура — DCGAN у найменшому вигляді: у генератора `ConvTranspose2d`, які
подвоюють сторону (7 → 14 → 28), у дискримінатора звичайні `Conv2d` із кроком 2,
які її вдвічі зменшують. `Tanh` на виході генератора означає, що картинки живуть у
відрізку від −1 до 1, тому справжні дані ми теж туди переносимо.

In [ ]:
LATENT = 16          # стільки випадкових чисел на вході генератора


class Generator(nn.Module):
    def __init__(self, latent=LATENT, width=24):
        super().__init__()
        self.width = width
        self.fc = nn.Linear(latent, width * 7 * 7)
        self.net = nn.Sequential(
            nn.BatchNorm2d(width), nn.ReLU(),
            nn.ConvTranspose2d(width, width // 2, 4, stride=2, padding=1),   # 7 → 14
            nn.BatchNorm2d(width // 2), nn.ReLU(),
            nn.ConvTranspose2d(width // 2, 1, 4, stride=2, padding=1),       # 14 → 28
            nn.Tanh())

    def forward(self, z):
        return self.net(self.fc(z).view(-1, self.width, 7, 7))


class MinibatchStd(nn.Module):
    # Канонічний рецепт проти колапсу: додати в дискримінатор канал,
    # у якому лежить розкид активацій ПО ПАКЕТУ. Пакет із однакових
    # картинок дає нуль, і дискримінатор може це помітити.
    def forward(self, x):
        spread = x.std(0, unbiased=False).mean()
        return torch.cat([x, spread.expand(x.size(0), 1, x.size(2), x.size(3))], dim=1)


class Discriminator(nn.Module):
    def __init__(self, width=24, minibatch_std=False):
        super().__init__()
        self.minibatch = MinibatchStd() if minibatch_std else None
        extra = 1 if minibatch_std else 0
        self.conv1 = nn.Conv2d(1, width // 2, 4, stride=2, padding=1)        # 28 → 14
        self.conv2 = nn.Conv2d(width // 2 + extra, width, 4, stride=2, padding=1)  # 14 → 7
        self.fc = nn.Linear(width * 7 * 7, 1)

    def forward(self, x):
        h = F.leaky_relu(self.conv1(x), 0.2)
        if self.minibatch is not None:
            h = self.minibatch(h)
        h = F.leaky_relu(self.conv2(h), 0.2)
        return self.fc(h.flatten(1))


generator_params = sum(p.numel() for p in Generator().parameters())
discriminator_params = sum(p.numel() for p in Discriminator().parameters())
discriminator_mb_params = sum(p.numel() for p in Discriminator(minibatch_std=True).parameters())
print("параметрів у генераторі        :", generator_params)
print("параметрів у дискримінаторі    :", discriminator_params)
print("те саме з minibatch std        :", discriminator_mb_params,
      "(+%d)" % (discriminator_mb_params - discriminator_params))
print("латентних чисел на вході       :", LATENT)

# minibatch std справді бачить розкид: на пакеті з однакових картинок він нуль
same = torch.zeros(8, 4, 6, 6)
varied = torch.randn(8, 4, 6, 6)
print("канал розкиду на однакових картинках: %.4f"
      % float(MinibatchStd()(same)[0, -1, 0, 0]))
print("канал розкиду на різних картинках   : %.4f"
      % float(MinibatchStd()(varied)[0, -1, 0, 0]))

## 6 · Навчання: два кроки в одному циклі

Один крок навчання GAN — це два оновлення підряд.

1. **Дискримінатор.** Показуємо йому пакет справжніх картинок із міткою «справжнє»
   і пакет підробок із міткою «підробка». Звичайна бінарна класифікація.
2. **Генератор.** Проганяємо ті самі підробки крізь дискримінатора ще раз (він уже
   змінився) і кажемо: мітка «справжнє». Дискримінатора при цьому не чіпаємо —
   градієнт іде крізь нього до генератора.

Оптимізатор — `Adam` із `betas=(0.5, 0.999)`. Перше число — це інерція градієнта:
за замовчуванням `0.9`, тобто оптимізатор довго памʼятає, куди йшов. У грі двох
мереж «куди йшов» застаріває щокроку, бо суперник змінився, і довга памʼять шкодить.
`0.5` — рецепт зі статті про DCGAN, і він справді потрібен.

Функція нижче зберігає **знімки** генератора на кількох епохах — щоб потім
відповісти на питання, чи колапс минає з часом.

In [ ]:
CHECKPOINTS = [8, 14, 20]


def train_gan(x_real, seed, width=24, lr_generator=2e-4, lr_discriminator=2e-4,
              minibatch_std=False, batch_size=64, latent=LATENT):
    torch.manual_seed(seed)
    generator = Generator(latent, width)
    discriminator = Discriminator(width, minibatch_std)
    opt_g = torch.optim.Adam(generator.parameters(), lr=lr_generator, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr_discriminator, betas=(0.5, 0.999))

    scaled = x_real * 2 - 1              # Tanh на виході, отже дані від −1 до 1
    count = len(scaled)
    bce = F.binary_cross_entropy_with_logits
    ones = torch.ones(batch_size, 1)
    zeros = torch.zeros(batch_size, 1)

    loss_history = []
    snapshots = {}
    for epoch in range(1, max(CHECKPOINTS) + 1):
        order = torch.randperm(count)
        sum_d = sum_g = 0.0
        batches = 0
        for start in range(0, count - batch_size + 1, batch_size):
            real = scaled[order[start:start + batch_size]]
            fake = generator(torch.randn(batch_size, latent))

            # крок 1: дискримінатор учиться відрізняти. detach() відрізає
            # градієнт від генератора — його ми тут не чіпаємо
            opt_d.zero_grad()
            loss_d = bce(discriminator(real), ones) + bce(discriminator(fake.detach()), zeros)
            loss_d.backward()
            opt_d.step()

            # крок 2: генератор бреше про мітку. Це та сама ненасичена втрата
            # -log D, з якої почався зошит
            opt_g.zero_grad()
            loss_g = bce(discriminator(fake), ones)
            loss_g.backward()
            opt_g.step()

            sum_d += loss_d.item()
            sum_g += loss_g.item()
            batches += 1

        loss_history.append((sum_d / batches, sum_g / batches))
        if epoch in CHECKPOINTS:
            generator.eval()
            with torch.no_grad():
                fixed = torch.Generator().manual_seed(5000 + seed)
                snapshots[epoch] = ((generator(torch.randn(600, latent, generator=fixed)) + 1) / 2).clone()
            generator.train()

    generator.eval()
    return generator, loss_history, snapshots


# дешева перевірка, що цикл узагалі крутиться: одна епоха на ста картинках
smoke_generator, smoke_losses, _ = train_gan(x_train[:128], seed=0)
print("пробний прогін: епох %d, остання втрата D %.4f, втрата G %.4f"
      % (len(smoke_losses), smoke_losses[-1][0], smoke_losses[-1][1]))
print("контрольні точки, на яких зберігаємо знімки:", CHECKPOINTS)

### Три конфігурації

Три точки на одній осі — **співвідношення сил генератора й дискримінатора**:

* **дискримінатор повільніший** — його швидкість навчання опущено вчетверо, до
  `5e-5`. Це найпоширеніша порада форумів: «дискримінатор надто сильний, послаб його»;
* **однакові кроки** — обидва мають `2e-4`. Це замовчування DCGAN;
* **генератор швидший** — швидкість генератора піднято до `5e-4`, дискримінатор
  лишається на `2e-4`.

Зверни увагу на пастку, заради якої взято саме ці три точки. Відношення
`lr_generator / lr_discriminator` дорівнює **4** у першій конфігурації, **1** у
другій і **2.5** у третій. Якщо справа у відношенні, перша й третя мають бути
схожими між собою й обидві — кращими за другу.

Кожна конфігурація йде на **трьох зернах**. Різниця, менша за розкид по зернах, не
є різницею — це правило курсу з теми
[26 «Одноетапні детектори»](../26-one-stage/lecture.html).

In [ ]:
CONFIGS = [
    ("дискримінатор повільніший", {"lr_discriminator": 5e-5}),
    ("однакові кроки", {}),
    ("генератор швидший", {"lr_generator": 5e-4}),
]

gan_started = time.perf_counter()
results = {}
for name, options in CONFIGS:
    results[name] = {"covered": {e: [] for e in CHECKPOINTS},
                     "fid": {e: [] for e in CHECKPOINTS},
                     "confidence": {e: [] for e in CHECKPOINTS},
                     "shares": {e: [] for e in CHECKPOINTS},
                     "midtones": {e: [] for e in CHECKPOINTS},
                     "losses": None, "samples": None}
    for seed in [0, 1, 2]:
        generator, losses, snapshots = train_gan(x_train, seed, **options)
        if seed == 0:
            results[name]["losses"] = losses
            results[name]["samples"] = snapshots[CHECKPOINTS[-1]]
        for epoch in CHECKPOINTS:
            covered, fid, confidence, shares = score(snapshots[epoch])
            results[name]["covered"][epoch].append(covered)
            results[name]["fid"][epoch].append(fid)
            results[name]["confidence"][epoch].append(confidence)
            results[name]["shares"][epoch].append(shares)
            results[name]["midtones"][epoch].append(midtone_share(snapshots[epoch]))
    final = results[name]["covered"][20]
    print("%-28s класів %.2f  %s" % (name, np.mean(final), final))

print()
print("девʼять GAN навчено за %.0f с" % (time.perf_counter() - gan_started))

## 7 · Головна таблиця теми

In [ ]:
LAST = CHECKPOINTS[-1]
print("%-28s %8s %14s %12s %12s" %
      ("конфігурація", "класів", "по зернах", "FID", "впевненість"))
print("-" * 80)
for name, _ in CONFIGS:
    data = results[name]
    print("%-28s %8.2f %14s %12.2f %12.4f" %
          (name, np.mean(data["covered"][LAST]), str(data["covered"][LAST]),
           np.mean(data["fid"][LAST]), np.mean(data["confidence"][LAST])))
print("-" * 80)
print("%-28s %8d %14s %12.4f %12.4f" %
      ("справжні фігури", 6, "[6, 6, 6]", real_baseline_fid, calibration_rows[0][3]))

In [ ]:
figure, axes = plt.subplots(len(CONFIGS), 12, figsize=(11, 4.4))
for row, (name, _) in enumerate(CONFIGS):
    samples = results[name]["samples"]
    for column in range(12):
        axes[row, column].imshow(samples[column, 0], cmap="gray", vmin=0, vmax=1)
        axes[row, column].axis("off")
    axes[row, 0].set_title(name, fontsize=8, loc="left")
plt.tight_layout()
plt.show()
print("по дванадцять зразків кожної конфігурації, зерно 0")

## 8 · Куди поділись фігури

Покриття — це одне число. Подивімось на самі частки: яких фігур генератор
намалював багато, а яких не намалював зовсім.

In [ ]:
print("%-28s %s" % ("конфігурація", "  ".join("%-9s" % n for n in SHAPE_NAMES)))
print("-" * 90)
for name, _ in CONFIGS:
    shares = results[name]["shares"][LAST][0]        # зерно 0
    print("%-28s %s" % (name, "  ".join("%-9.2f" % v for v in shares)))
print("-" * 90)
print("%-28s %s" % ("рівномірно було б", "  ".join("%-9.2f" % (1 / 6) for _ in range(6))))

In [ ]:
for name, _ in CONFIGS:
    print(name)
    for seed in range(3):
        shares = results[name]["shares"][LAST][seed]
        print("   зерно %d: класів %d  FID %7.2f  впевненість %.4f  напівтони %.4f  %s"
              % (seed, results[name]["covered"][LAST][seed],
                 results[name]["fid"][LAST][seed],
                 results[name]["confidence"][LAST][seed],
                 results[name]["midtones"][LAST][seed],
                 " ".join("%.3f" % v for v in shares)))
print()
print("порядок класів:", " ".join(SHAPE_NAMES))

Тепер те саме, але усереднено по всіх трьох зернах — щоб було видно, що порядок
вимирання класів не випадковість одного прогону.

In [ ]:
print("%-11s %s" % ("фігура", "  ".join("%-11s" % n[:11] for n, _ in CONFIGS)))
print("-" * 60)
average_shares = {}
for name, _ in CONFIGS:
    average_shares[name] = np.mean(np.array(results[name]["shares"][LAST]), axis=0)
for kind in range(6):
    row = "  ".join("%-11.3f" % average_shares[name][kind] for name, _ in CONFIGS)
    print("%-11s %s" % (SHAPE_NAMES[kind], row))

## 9 · Колапс чи просто недонавчання?

Питання, яке треба поставити перш ніж заявляти про колапс мод: а може, генератор
просто ще не встиг навчитись? Знімки, збережені на трьох епохах, відповідають прямо.

In [ ]:
print("%-28s %s" % ("конфігурація", "  ".join("епоха %-2d " % e for e in CHECKPOINTS)))
print("-" * 62)
for name, _ in CONFIGS:
    row = "  ".join("%-9.2f" % np.mean(results[name]["covered"][e]) for e in CHECKPOINTS)
    print("%-28s %s" % (name, row))
print()
print("FID (менше — краще):")
for name, _ in CONFIGS:
    row = "  ".join("%-9.1f" % np.mean(results[name]["fid"][e]) for e in CHECKPOINTS)
    print("%-28s %s" % (name, row))

## 10 · Криві втрат нічого не кажуть

А тепер найважливіша відмінність GAN від усього, що було в курсі досі. Візьмемо
криві втрат обох мереж для трьох конфігурацій із дуже різним покриттям — і
подивимось, чи можна за ними хоч щось вгадати.

In [ ]:
figure, axes = plt.subplots(1, len(CONFIGS), figsize=(13, 2.9), sharey=True)
for column, (name, _) in enumerate(CONFIGS):
    losses = np.array(results[name]["losses"])
    axes[column].plot(range(1, len(losses) + 1), losses[:, 0], label="дискримінатор")
    axes[column].plot(range(1, len(losses) + 1), losses[:, 1], label="генератор")
    axes[column].set_title("%s\nкласів: %d" % (name, results[name]["covered"][LAST][0]), fontsize=9)
    axes[column].set_xlabel("епоха")
axes[0].set_ylabel("втрата")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("%-28s %10s %10s %10s %8s" %
      ("конфігурація", "втрата D", "втрата G", "розмах G", "класів"))
print("-" * 72)
for name, _ in CONFIGS:
    losses = np.array(results[name]["losses"])
    tail = losses[-5:]
    print("%-28s %10.4f %10.4f %10.4f %8d" %
          (name, tail[:, 0].mean(), tail[:, 1].mean(),
           losses[:, 1].max() - losses[:, 1].min(), results[name]["covered"][LAST][0]))

In [ ]:
print("епоха  " + "  ".join("%-19s" % n[:19] for n, _ in CONFIGS))
print("       " + "  ".join("%-19s" % "  втрата D / втрата G" for _ in CONFIGS))
print("-" * 92)
for epoch_index in range(len(results[CONFIGS[0][0]]["losses"])):
    row = "  ".join("%8.4f %8.4f  " % tuple(results[name]["losses"][epoch_index])
                    for name, _ in CONFIGS)
    print("%-6d %s" % (epoch_index + 1, row))

## 11 · Чому першим гине кільце

Кільце зникає в усіх конфігураціях. Гіпотеза: справа не у випадковості, а в тому,
що кільце **найважче намалювати** — воно найтонше, і невелика неохайність його
знищує.

Перевіримо це прямо, без жодного GAN. Візьмемо справжні фігури й зіпсуємо їх
однаково для всіх класів: розмиємо кожну однаковим вікном. Потім спитаємо суддю,
яку частку кожного класу він ще впізнає. Якщо гіпотеза правильна, кільце обвалиться
першим.

In [ ]:
@torch.no_grad()
def recognised_share(images, labels, blur_radius):
    # Розмиваємо всі картинки однаково й дивимось, який клас суддя ще впізнає.
    if blur_radius > 0:
        window = 2 * blur_radius + 1
        images = F.avg_pool2d(images, window, stride=1, padding=blur_radius)
    predicted = judge(images).argmax(1)
    return [float((predicted[labels == kind] == kind).float().mean()) for kind in range(6)]


print("%-9s %s" % ("розмиття", "  ".join("%-11s" % n[:11] for n in SHAPE_NAMES)))
print("-" * 78)
fragility = {}
for blur_radius in [0, 1, 2, 3]:
    shares = recognised_share(x_test, y_test, blur_radius)
    fragility[blur_radius] = shares
    print("%-9d %s" % (blur_radius, "  ".join("%-11.3f" % v for v in shares)))

print()
worst = int(np.argmin(fragility[1]))
print("на розмитті 1 найгірше впізнається:", SHAPE_NAMES[worst], "— %.3f" % fragility[1][worst])
print("а частка яскравих пікселів у ньому найменша:", "%.4f" % bright_share[worst],
      "проти %.4f у найтовщої фігури" % max(bright_share))

## 12 · GAN проти VAE однією метрикою

Тема [37](../37-autoencoders/lecture.html) показала VAE: він теж генерує картинки з
випадкового вектора, але вчиться зовсім інакше — мінімізує **похибку відновлення**.
Порівняймо їх однією метрикою й на тому самому датасеті.

Щоб порівняння було чесним, VAE дістає **рівно стільки ж епох**, скільки GAN, —
двадцять, і той самий латентний простір на 16 чисел. Коефіцієнт `beta`, який
керує компромісом VAE, беремо 0.2 — це його власна ручка, і про неї йдеться в
темі 37.

Додаємо третє число — **різкість**. Міряємо її просто: частка пікселів, які ані
чорні, ані білі (значення між 0.25 і 0.75). У справжніх фігур таких майже немає —
межа різка. У розмитої картинки їх багато.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent=LATENT, width=24):
        super().__init__()
        self.width = width
        self.encoder = nn.Sequential(
            nn.Conv2d(1, width // 2, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(width // 2, width, 4, stride=2, padding=1), nn.ReLU(), nn.Flatten())
        self.to_mean = nn.Linear(width * 49, latent)
        self.to_logvar = nn.Linear(width * 49, latent)
        self.fc = nn.Linear(latent, width * 7 * 7)
        self.decoder = nn.Sequential(
            nn.ReLU(), nn.ConvTranspose2d(width, width // 2, 4, stride=2, padding=1),
            nn.ReLU(), nn.ConvTranspose2d(width // 2, 1, 4, stride=2, padding=1), nn.Sigmoid())

    def decode(self, z):
        return self.decoder(self.fc(z).view(-1, self.width, 7, 7))

    def forward(self, x):
        hidden = self.encoder(x)
        mean, logvar = self.to_mean(hidden), self.to_logvar(hidden)
        z = mean + torch.randn_like(mean) * torch.exp(0.5 * logvar)
        return self.decode(z), mean, logvar


def train_vae(images, seed, epochs=20, batch_size=64, lr=2e-3, beta=0.2):
    torch.manual_seed(seed)
    model = VAE()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    count = len(images)
    for epoch in range(epochs):
        order = torch.randperm(count)
        for start in range(0, count - batch_size + 1, batch_size):
            batch = images[order[start:start + batch_size]]
            optimizer.zero_grad()
            output, mean, logvar = model(batch)
            # похибка відновлення — та сама явна ціль, якої в GAN немає
            reconstruction = F.mse_loss(output, batch, reduction="sum") / len(batch)
            # розходження Кульбака–Лейблера: тягне хмарку кодів до N(0, 1)
            divergence = -0.5 * (1 + logvar - mean * mean - logvar.exp()).sum() / len(batch)
            (reconstruction + beta * divergence).backward()
            optimizer.step()
    model.eval()
    return model


vae_started = time.perf_counter()
vae_rows = []
for seed in [0, 1, 2]:
    vae = train_vae(x_train, seed)
    with torch.no_grad():
        fixed = torch.Generator().manual_seed(5000 + seed)
        samples = vae.decode(torch.randn(600, LATENT, generator=fixed))
    covered, fid, confidence, _ = score(samples)
    vae_rows.append((covered, fid, confidence, midtone_share(samples)))
    if seed == 0:
        vae_samples = samples
print("три VAE навчено за %.0f с" % (time.perf_counter() - vae_started))
for seed, row in enumerate(vae_rows):
    print("VAE зерно %d: класів %d  FID %.2f  впевненість %.4f  напівтони %.4f" % (seed, *row))

In [ ]:
gan_best = "генератор швидший"
gan_samples = results[gan_best]["samples"]
gan_plain_samples = results["однакові кроки"]["samples"]

print("%-18s %8s %10s %12s %12s" %
      ("модель", "класів", "FID", "напівтони", "впевненість"))
print("-" * 66)
print("%-18s %8.2f %10.2f %12.4f %12.4f" %
      ("справжні", 6.0, real_baseline_fid, midtone_share(x_test), calibration_rows[0][3]))
print("%-18s %8.2f %10.2f %12.4f %12.4f" %
      ("VAE", np.mean([r[0] for r in vae_rows]), np.mean([r[1] for r in vae_rows]),
       np.mean([r[3] for r in vae_rows]), np.mean([r[2] for r in vae_rows])))
print("%-18s %8.2f %10.2f %12.4f %12.4f" %
      ("GAN (найкращий)", np.mean(results[gan_best]["covered"][LAST]),
       np.mean(results[gan_best]["fid"][LAST]),
       np.mean(results[gan_best]["midtones"][LAST]),
       np.mean(results[gan_best]["confidence"][LAST])))
print("%-18s %8.2f %10.2f %12.4f %12.4f" %
      ("GAN (базовий)", np.mean(results["однакові кроки"]["covered"][LAST]),
       np.mean(results["однакові кроки"]["fid"][LAST]),
       np.mean(results["однакові кроки"]["midtones"][LAST]),
       np.mean(results["однакові кроки"]["confidence"][LAST])))
print()
print("покриття по зернах: VAE %s, GAN найкращий %s, GAN базовий %s"
      % ([r[0] for r in vae_rows], results[gan_best]["covered"][LAST],
         results["однакові кроки"]["covered"][LAST]))
print("напівтони по зернах: VAE %s, GAN найкращий %s"
      % ([round(r[3], 4) for r in vae_rows],
         [round(v, 4) for v in results[gan_best]["midtones"][LAST]]))

In [ ]:
figure, axes = plt.subplots(3, 10, figsize=(11, 3.6))
for column in range(10):
    axes[0, column].imshow(x_test[column, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, column].imshow(vae_samples[column, 0], cmap="gray", vmin=0, vmax=1)
    axes[2, column].imshow(gan_samples[column, 0], cmap="gray", vmin=0, vmax=1)
    for row in range(3):
        axes[row, column].axis("off")
axes[0, 0].set_title("справжні", fontsize=8, loc="left")
axes[1, 0].set_title("VAE", fontsize=8, loc="left")
axes[2, 0].set_title("GAN", fontsize=8, loc="left")
plt.tight_layout()
plt.show()
print("напівтонів у справжніх: %.4f" % midtone_share(x_test))
print("напівтонів у VAE      : %.4f" % midtone_share(vae_samples))
print("напівтонів у GAN      : %.4f" % midtone_share(gan_samples))
print("тобто GAN різкіший за VAE у %.1f раза"
      % (midtone_share(vae_samples) / midtone_share(gan_samples)))

## 13 · Що зошит показав

In [ ]:
print("1. Ненасичена втрата дає при D = 0.0180 градієнт у %.1f раза більший."
      % (saturation_table[0][3] / saturation_table[0][2]))
print("2. Суддя на справжніх фігурах: %.4f. Базова лінія FID: %.4f."
      % (judge_accuracy, real_baseline_fid))
print("3. Метрика ставить повний колапс («лише кола») у %.0f разів гірше за базову лінію."
      % (circles_only_fid / real_baseline_fid))
print("4. Базовий GAN покриває %.2f класи з шести; купа по зернах %s."
      % (np.mean(results["однакові кроки"]["covered"][LAST]),
         results["однакові кроки"]["covered"][LAST]))
print("5. Сповільнення дискримінатора зробило гірше: %.2f, купа %s."
      % (np.mean(results["дискримінатор повільніший"]["covered"][LAST]),
         results["дискримінатор повільніший"]["covered"][LAST]))
print("6. Прискорення генератора допомогло: %.2f, купа %s — купи не перетинаються."
      % (np.mean(results["генератор швидший"]["covered"][LAST]),
         results["генератор швидший"]["covered"][LAST]))
print("7. Найвпевненіший суддя — на найгіршій конфігурації: %.4f проти %.4f у найкращої."
      % (np.mean(results["дискримінатор повільніший"]["confidence"][LAST]),
         np.mean(results["генератор швидший"]["confidence"][LAST])))
print()
print("час усього зошита: %.0f с" % (time.perf_counter() - notebook_started))

## Завдання

### 🟢 Рівень 1
Додай до таблиці калібрування ще один завідомо поганий набір: **лише кільця**.
Порівняй його FID із «лише кола». Чи однаково метрика карає колапс на різні класи?

**Зроблено, якщо:** у таблиці зʼявився новий рядок і ти можеш сказати, який із двох
колапсів метрика вважає гіршим і чому.

### 🟡 Рівень 2
Прожени конфігурацію «генератор швидший» ще з двома значеннями `lr_generator`
(наприклад `3e-4` і `1e-3`) на тих самих трьох зернах. Побудуй залежність покриття
від швидкості генератора.

**Зроблено, якщо:** є таблиця з трьох-чотирьох точок із розкидом по зернах, і ти
можеш сказати, чи це монотонна залежність, чи в неї є максимум.

### 🔴 Рівень 3
Заміни втрату генератора на **насичену** — `log(1 - D)` замість `-log D` — і навчи
GAN у конфігурації «однакові кроки» на трьох зернах.

**Зроблено, якщо:** ти показуєш, що сталося з покриттям і з кривою втрат, і
повʼязуєш це з таблицею градієнтів із першого розділу зошита.